# HyDE: Hypothetical Document Embeddings for Advanced RAG

This notebook explains and demonstrates HyDE, short for Hypothetical Document Embeddings, and compares it directly against naive retrieval so the difference in retrieval quality is easy to see.

## What problem does this solve?

In a normal RAG pipeline, a user question is embedded directly, and that embedding is compared against the embeddings of the document chunks stored in a vector database. This works reasonably well, but it has a structural weakness. Questions and answers are often written in very different styles. A question is typically short, phrased as an inquiry, and may not share much vocabulary with the passage that actually contains the answer. Since embedding models are trained to place semantically similar text close together, a question embedding and an answer embedding do not always end up close together in vector space, even when the answer is exactly what the question is looking for.

HyDE addresses this mismatch with a clever trick. Instead of embedding the question itself, first ask the language model to imagine what a good answer to the question would look like, without checking anything against the real documents yet. This imagined answer is called the hypothetical document. It is fictional and may contain inaccurate details, but it is written in the same style and vocabulary as a real answer would be. That hypothetical document is then embedded and used to search the vector database. Because the hypothetical document reads like an answer rather than like a question, its embedding tends to land much closer to the real answer passages stored in the database, which often improves retrieval quality.

## What this notebook covers

1. Installing dependencies and configuring API access.
2. Initializing the language model and the embedding model.
3. Loading a source document from the web and splitting it into chunks.
4. Building an in memory vector store from those chunks.
5. Generating a hypothetical document for a sample question using the language model.
6. Retrieving documents twice, once using the hypothetical document and once using the raw question, so the two retrieval strategies can be compared.
7. Formatting the retrieved context and generating a final answer for each retrieval strategy.
8. A diagram summarizing the complete HyDE pipeline end to end.
9. A conclusion comparing HyDE retrieval against naive retrieval and summarizing what was observed.


## Section 1. Installing dependencies

This notebook depends on three package groups.

1. `langchain google genai` provides the LangChain wrappers around Google's Gemini chat model and embedding models.
2. `langchain community` provides community maintained integrations, including the web page loader and the HuggingFace embeddings wrapper used later in this notebook.
3. `chromadb` provides Chroma, a lightweight vector database that can run entirely in memory, which is used here to store and search the document chunk embeddings.


In [ ]:
!pip install langchain-google-genai langchain-community chromadb

## Section 2. API configuration and imports

Before any model can be called, the required classes need to be imported, and the API key used to authenticate with Google's Gemini service needs to be loaded. Loading the key from a local `.env` file rather than typing it directly into the notebook keeps the real key out of the notebook file itself, which matters if this notebook is ever pushed to a public repository such as GitHub.


### Core imports

1. `os` is used to read and set environment variables, such as the API key.
2. `warnings` is used purely to silence noisy, non critical warning messages so the notebook output stays readable.
3. `WebBaseLoader` downloads and parses a web page into LangChain `Document` objects.
4. `RecursiveCharacterTextSplitter` splits long text into smaller overlapping chunks, which is the standard chunking approach used before embedding text for retrieval.
5. `Chroma` is the vector store integration that stores chunk embeddings and performs similarity search over them.
6. `ChatPromptTemplate` builds reusable prompt templates with placeholders that get filled in at request time, used here both for generating the hypothetical document and for generating the final answer.


In [ ]:
import os
import warnings
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate

warnings.filterwarnings("ignore")

### Loading the API key from the environment

`load_dotenv` reads a local `.env` file (which should never be committed to source control) and loads any variables defined in it into the process environment. The Gemini API key is then read from that environment and set explicitly as `GOOGLE_API_KEY`, since that is the exact environment variable name the Gemini integration expects to find. Keeping the key in a `.env` file, rather than hard coded as a string, means the notebook can safely be shared or pushed to GitHub without leaking the real credential.


In [ ]:
from dotenv import load_dotenv

# Load variables from .env
load_dotenv()

# Get the API key
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

## Section 3. Model initialization

Two different models are required for this pipeline, exactly as in most RAG setups.

1. A language model, used both to generate the hypothetical document and to generate the final answer once relevant context has been retrieved. This notebook uses Google's `gemini 2.5 flash` model.
2. An embedding model, used to turn text, whether it is the hypothetical document, the raw question, or a document chunk, into a numeric vector for similarity search. This notebook uses a HuggingFace BGE embedding model, `BAAI/bge small en v1.5`.


### Initializing the chat model

`ChatGoogleGenerativeAI` wraps Gemini so it can be called through LangChain's standard chat interface. This is the single model used everywhere a natural language generation step is needed later in the notebook.


In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

### A quick sanity check

Before building the rest of the pipeline, it is worth confirming the model is reachable and responding correctly. Asking it directly what HyDE is serves as both a connectivity test and a light introduction to the technique from the model's own perspective.


In [ ]:
result = llm.invoke("What is Hyde(Hypothetical Document Embedding)?")
print(result.content)

### Initializing the embedding model

`HuggingFaceBgeEmbeddings` wraps a BGE family sentence embedding model from HuggingFace. BGE models are specifically trained to produce embeddings well suited for retrieval tasks, which makes them a solid choice for the similarity search performed later in this notebook.


In [ ]:
from langchain_community.embeddings import HuggingFaceBgeEmbeddings

### Configuring the embedding model

`normalize_embeddings` is set to true so that every embedding vector produced by the model has unit length. Normalizing vectors this way means a simple dot product between two embeddings is equivalent to computing their cosine similarity, which is the standard similarity measure used for comparing text embeddings.


In [ ]:
model_name = "BAAI/bge-small-en-v1.5"
encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity

### Creating the embedding function and testing it

The embedding model is instantiated with the configuration above, and then immediately tested on a short sample string. This confirms the model loads correctly and produces a usable embedding before it gets used anywhere else in the pipeline. The commented out `model_kwargs` line shows how a GPU device could be specified if one were available, since embedding is much faster on a GPU than on a CPU.


In [ ]:
embedding_function = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    #model_kwargs={'device': 'cuda'},
    encode_kwargs=encode_kwargs,
)

vector = embedding_function.embed_query("Hello, how are you?")

### Inspecting the embedding vector

Printing the length of the vector confirms the dimensionality of the embedding space this model produces, and printing the first few values gives a concrete sense of what an embedding actually looks like under the hood, namely a list of floating point numbers with no inherent meaning on their own, only meaning relative to other embeddings in the same space.


In [ ]:
print(len(vector))

In [ ]:
print(vector[:5])

## Section 4. Data loading and splitting

With both models ready, the next step is to bring in the source text that will be searched. This notebook loads a blog post about LLM powered autonomous agents, which covers prompting techniques including Chain of Thought, making it a good fit for testing questions about prompting strategies later on.


### Loading the source page

`WebBaseLoader` downloads the given URL and parses it into one or more LangChain `Document` objects, each containing the extracted text along with basic metadata such as the source URL.


In [ ]:
# Loading data from the blog link mentioned in the video
loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
data = loader.load()

Checking how many documents were loaded is a simple sanity check. A single web page typically loads as a single `Document` object containing the full page text.


In [ ]:
len(data)

### Splitting the document into chunks

A single blog post is far too long to embed as one unit, since a very long chunk would dilute the embedding and make similarity search much less precise. `RecursiveCharacterTextSplitter` breaks the text into smaller chunks of a target size, while trying to split on natural boundaries such as paragraphs and sentences wherever possible. Here each chunk targets 300 characters, with an overlap of 50 characters between consecutive chunks so that context is not abruptly cut off exactly at a chunk boundary.


In [ ]:
# Splitting data into chunks of 300 with an overlap of 50
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
splits = text_splitter.split_documents(data)

Checking the number of resulting chunks confirms the split actually happened and gives a sense of scale, namely how many separate pieces the single long blog post was broken into.


In [ ]:
len(splits)

Printing one of the chunks directly shows exactly what a single unit of retrievable text looks like once splitting is complete, which is useful context for interpreting the retrieval results produced later in the notebook.


In [ ]:
print(splits[1])

## Section 5. Vector store setup

Once the chunks exist, they need to be embedded and stored somewhere that supports fast similarity search. This notebook uses Chroma, run entirely in memory, which is convenient for experimentation since it requires no separate server or deployment step.


`Chroma.from_documents` takes the list of chunks along with the embedding function, computes an embedding for every chunk, and stores them all inside an in memory Chroma collection. `as_retriever` then wraps that vector store in a retriever interface, which exposes a simple method for fetching the most similar chunks to any given piece of query text.


In [ ]:
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding_function)
retriever = vectorstore.as_retriever()

## Section 6. HyDE: generating the hypothetical document

This is the core step that distinguishes HyDE from a normal RAG pipeline. Instead of immediately searching the vector store with the raw user question, the language model is first asked to write a hypothetical answer to that question, without consulting the real documents at all.

The prompt below explicitly instructs the model to produce only the hypothetical answer and nothing else, since any extra commentary from the model would just add noise to the text that is about to be embedded and searched against.


In [ ]:
# User question
question = "What are the different Chain of Thought prompting?"

# Prompt template for the hypothetical answer
template = """For the given question, try to generate a hypothetical answer. 
Only generate the answer and nothing else.
Question: {question}"""

prompt_hyde = ChatPromptTemplate.from_template(template)

# Generate the hypothetical document
chain_hyde = prompt_hyde | llm
hypothetical_answer = chain_hyde.invoke({"question": question})
print("Hypothetical Document Generated:")
print(hypothetical_answer.content)

The `chain_hyde = prompt_hyde | llm` line builds a small LangChain pipeline using the pipe operator. The prompt template is filled in with the question, and the resulting text is immediately passed into the language model, producing the hypothetical answer shown above. Note that this hypothetical answer is generated entirely from the model's own general knowledge. It has not been checked against the real blog post yet, so it may be incomplete or even factually off in places. Its only job is to read like a plausible answer, so that its embedding lands close to real answer passages in vector space.


## Section 7. Retrieving documents: HyDE versus naive retrieval

With the hypothetical document generated, two separate retrieval calls are made against the same vector store, so the two strategies can be compared directly using the exact same underlying data and embedding model.


### Retrieval based on the hypothetical answer

Here the retriever is queried using the hypothetical document generated in the previous section, rather than the original question. Because the hypothetical document is written in the style of an answer, its embedding is expected to land closer to the real answer passages in the vector store than the original question's embedding would.


In [ ]:
# 1. Retrieval based on Hypothetical Answer
similar_document_1 = retriever.get_relevant_documents(hypothetical_answer.content)
similar_document_1

In [ ]:
similar_document_1[0].page_content

### Retrieval based on the raw question

For comparison, the same retriever is queried a second time, this time using the original user question directly, exactly as a naive RAG pipeline would do it, with no hypothetical document step at all.


In [ ]:
# 2. Retrieval based on Direct Query (Naive)
similar_document_2 = retriever.get_relevant_documents(question)
similar_document_2

In [ ]:
similar_document_2[0].page_content

## Section 8. Formatting context and generating the final answers

Each retrieval call returns a list of separate document chunks. Before these can be handed to the language model as context, they need to be merged into a single block of text. A small helper function, `format_docs`, joins the page content of every retrieved chunk together, separated by blank lines, producing one combined context string per retrieval strategy.


In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

formatted_doc_1 = format_docs(similar_document_1)
formatted_doc_2 = format_docs(similar_document_2)

Printing each formatted context string shows exactly what text will actually be handed to the language model for each retrieval strategy, which makes it easy to see up front whether the HyDE retrieved context or the naive retrieved context looks more relevant to the original question.


In [ ]:
formatted_doc_1

In [ ]:
formatted_doc_2

### Generating the final answer for each strategy

A second prompt template is used here, distinct from the one used to generate the hypothetical document. This template asks the language model to answer the original question strictly based on the given context, which is exactly how a normal RAG answer generation step works. The same question is answered twice, once using the HyDE retrieved context and once using the naive retrieved context, so the final answers themselves can be compared side by side.


In [ ]:
# Final prompt template
template_rag = """Please answer the question based on the given context.
Context: {context}
Question: {question}"""

prompt_rag = ChatPromptTemplate.from_template(template_rag)

# Comparison of responses
print("--- Response based on HyDE Retrieval ---")
final_prompt_1 = prompt_rag.format(context=formatted_doc_1, question=question)
response_1 = llm.invoke(final_prompt_1)
print(response_1.content)

In [ ]:
print("\n--- Response based on Naive Retrieval ---")
final_prompt_2 = prompt_rag.format(context=formatted_doc_2, question=question)
response_2 = llm.invoke(final_prompt_2)
print(response_2.content)

## Section 9. The complete HyDE pipeline

The diagram below summarizes every step performed above, from the original user question all the way through to the final generated answer, in the order those steps actually happen in a HyDE pipeline.

```text
User Question
      |
      v
LLM generates Hypothetical Document
      |
      v
Embedding Model
      |
      v
Vector Database
      |
      v
Retriever
      |
      v
Retrieved Documents
      |
      v
format_docs()
      |
      v
One Context String
      |
      v
Prompt Template
      |
      v
LLM
      |
      v
Final Answer
```

Reading this diagram from top to bottom, the key detail that makes this HyDE rather than a normal RAG pipeline is the second box. The language model generates a hypothetical document before anything is embedded or searched, and it is that hypothetical document, not the raw user question, which flows into the embedding model and the vector database.


## Conclusion

This notebook built a single shared vector store and a single shared language model, then compared two different ways of querying that store for the same question, `What are the different Chain of Thought prompting?`.

Naive retrieval embedded the raw user question directly and searched the vector store with it. HyDE retrieval instead first asked the language model to imagine a plausible answer to the question, and embedded and searched using that hypothetical answer instead of the question itself.

Since questions and answers are often phrased very differently, even when they are about the exact same topic, a raw question embedding does not always land close to the embeddings of the passages that actually answer it. The hypothetical document generated by HyDE is written in the same style as a real answer, so its embedding tends to land closer to genuinely relevant passages in vector space, which can produce retrieved context that is more directly useful to the final answer generation step.

Comparing the two printed responses above, from the HyDE based retrieval and from the naive retrieval, shows concretely whether this effect held for this particular question and this particular source document. The main idea to take away is that the quality of a RAG answer depends heavily on what text is actually used to search the vector store, and that text does not have to be the raw user question. Generating a better search query first, as HyDE does by imagining an answer, is one practical way to improve retrieval quality without changing the embedding model, the vector store, or the chunking strategy at all.

Some natural next steps for extending this notebook include testing HyDE across a larger and more varied set of questions rather than a single example, trying multiple hypothetical documents per question and averaging their embeddings, combining HyDE with a reranking step, and measuring retrieval quality with a quantitative metric rather than only reading the final answers side by side.
